## Show strength on abundance & expression

In [ ]:
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from adjustText import adjust_text
from dynaconf import Dynaconf
from tqdm import tqdm

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

# ---- Scanpy global settings ----
sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(
    dpi=150,
    dpi_save=300,
    format="pdf",
    facecolor="white",
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(4, 4),
    transparent=True,
)

# ---- Matplotlib global settings ----
mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.fontsize": 6,
        "axes.titlesize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

## Load data

In [ ]:
# Load number of DE features per perturbation and cell type
tf_expression_measure_df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "tf_ko_panel_contrastiveVI_de_summary.tsv", sep="\t")

# Load differential abundance measures per perturbation and cell type
tf_abundance_measure_df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "tf_ko_panel_contrastiveVI_da_summary.tsv", sep="\t")

# Rename columns and format cell type names
tf_abundance_measure_df = tf_abundance_measure_df.rename(columns={"Cell Type": "cell_type", "condition": "perturbed_tf"})
tf_abundance_measure_df["cell_type"] = tf_abundance_measure_df["cell_type"].str.replace(" ", "_").str.replace("/", "_")

# Compute total number of DE features per perturbation and cell type
tf_expression_measure_df = tf_expression_measure_df.assign(total = lambda x: x["Up"] + x["Down"]).rename(columns={"test_condition": "perturbed_tf", "group": "cell_type"})[["perturbed_tf", "cell_type", "total"]]

# Subset to measure of effect size (total DE genes & absolute DA log fold change)
tf_abundance_measure_df = tf_abundance_measure_df[["perturbed_tf", "cell_type", "abs_log2_fc"]]

# Log scale number of total DE genes
tf_expression_measure_df["total"] = np.log10(tf_expression_measure_df["total"] + 1)

In [ ]:
tf_expression_measure_df.head()

In [ ]:
len(tf_expression_measure_df["perturbed_tf"].unique()), len(tf_abundance_measure_df["perturbed_tf"].unique())

In [ ]:
# Merge the two dataframes
merged_df = pd.merge(
    tf_expression_measure_df,
    tf_abundance_measure_df,
    on=["perturbed_tf", "cell_type"],
    how="outer"
)

# Map cell types to eecs vs non-eecs
celltype_to_eec = {
    "EC_Cells": "EEC",
    "X_Cells": "EEC",
    "D_Cells": "EEC",
    "I_N_Cells": "EEC",
    "K_Cells": "EEC",
    "EEC_Progenitors": "EEC",

    "Enterocytes": "Non-EEC",
    "Goblet_Cells": "Non-EEC",
    "Stem_Cells": "Non-EEC",
    "TA_Cells": "Non-EEC",
}
merged_df["cell_type_group"] = merged_df["cell_type"].map(celltype_to_eec)
merged_df["total"] = merged_df["total"].fillna(0)
merged_df["log2-fold change"] = merged_df["abs_log2_fc"].fillna(0)

In [ ]:
merged_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Choose threshold for labeling ----
# You can adjust these depending on the spread of your data
logfc_threshold = 1.0               # label points with |log2FC| > 1
deg_threshold = merged_df["total"].quantile(0.90)   # label top 10% DE gene counts

plt.figure(figsize=(5, 4))

# Scatterplot
sns.scatterplot(
    data=merged_df,
    x="total",
    color="black",
    y="log2-fold change",
    s=50,
    edgecolor="black",
)

plt.axhline(0, color="grey", linestyle="--", linewidth=1.2)

plt.xlabel("Number of Differentially Expressed Genes", fontsize=12)
plt.ylabel("Log2 Fold Change in Abundance", fontsize=12)
plt.grid(False)
plt.tight_layout()
plt.savefig(Path(sc.settings.figdir) / "DA_vs_DE_per_TF_per_annotation.pdf")

In [ ]:
merged_df["number_de_genes"] = merged_df["total"]
merged_df["abundance_log2fc"] = merged_df["log2-fold change"].abs()


# Collapse per TF to max 
merged_summary_df = merged_df.groupby(["perturbed_tf", "cell_type_group"]).agg({
    "number_de_genes": "max",
    "abundance_log2fc": "max"
}).reset_index()

In [ ]:
merged_summary_df.query('perturbed_tf == "Tox3"')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

groups = ["Non-EEC", "EEC"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False, sharey=True)

for ax, group in zip(axes, groups):
    df_sub = merged_summary_df[merged_summary_df["cell_type_group"] == group]
    
    # Label points
    for _, row in df_sub.iterrows():
        if row["abundance_log2fc"] > 0:
            ax.text(
                row["number_de_genes"],
                row["abundance_log2fc"],
                row["perturbed_tf"],
                fontsize=9,
                ha="left",
                va="bottom",
            )


    sns.scatterplot(
        data=df_sub,
        x="number_de_genes",
        y="abundance_log2fc",
        s=50,
        color="black",
        edgecolor="black",
        linewidth=0.6,
        ax=ax,
    )
    ax.set_title(group, fontsize=14)
    ax.set_xlabel("Max Number of log10(DE genes)\n(State Change)", fontsize=12)
    ax.set_ylabel("Max log2FC in proportions\n(Fate Change)", fontsize=12)
    ax.grid(False)
plt.tight_layout()
plt.savefig(Path(sc.settings.figdir) / "DA_vs_DE_per_TF_per_cell_type_group.pdf")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import scanpy as sc

groups = ["Non-EEC", "EEC"]

# choose TFs to label manually
label_tfs = ["Atoh1", "Hes1", "Rora", "Id3", "Lmx1b", "Hif1a", "Fos", "Rfx3", "Cxxc4", "Id3", "Hes1", "Znf800", "Spib", "Pax4", "Percc1", "Prdm16", "Insm1", "Rfx6", "Lmx1b", "Sox4"]  # edit this

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False, sharey=True)

for ax, group in zip(axes, groups):
    df_sub = merged_summary_df[
        merged_summary_df["cell_type_group"] == group
    ].copy()

    # density background
    sns.kdeplot(
        data=df_sub,
        x="number_de_genes",
        y="abundance_log2fc",
        fill=True,
        levels=20,
        thresh=0.05,
        color="lightgray",
        alpha=0.6,
        ax=ax,
    )

    # points on top
    sns.scatterplot(
        data=df_sub,
        x="number_de_genes",
        y="abundance_log2fc",
        s=50,
        color="black",
        edgecolor="black",
        linewidth=0.6,
        alpha=0.8,
        ax=ax,
        zorder=3,
    )

    # label selected TFs
    df_label = df_sub[df_sub["perturbed_tf"].isin(label_tfs)]

    for _, row in df_label.iterrows():
        ax.text(
            row["number_de_genes"],
            row["abundance_log2fc"],
            row["perturbed_tf"],
            fontsize=9,
            ha="left",
            va="bottom",
            color="black",
            zorder=4,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                edgecolor="black",
                linewidth=1,
                alpha=0.9,
            ),
        )

    ax.axhline(0, linestyle="--", linewidth=0.8, color="gray")
    ax.set_title(group, fontsize=14)
    ax.set_xlabel("Max Number of log10(DE genes)\n(State Change)", fontsize=12)
    ax.set_ylabel("Max log2FC in proportions\n(Fate Change)", fontsize=12)
    ax.grid(False)

plt.ylim(0, None)  # focus on positive log2FCs
plt.tight_layout()
plt.savefig(Path(sc.settings.figdir) / "DA_vs_DE_per_TF_per_cell_type_group.pdf")
plt.show()